# Kilimo ADTC 2026 — Colab GPU train → GGUF → Hugging Face

1. Runtime → Change runtime type → **T4 GPU** (or better)
2. Set `HF_TOKEN` (write access) in the next cell
3. Runtime → Run all

Uploads to `rssebambulidde/adtc-kilimo-0.5b-gguf` as `adtc-kilimo-0.5b-q4_k_m.gguf`.

In [ ]:
# Paste a Hugging Face write token, or use Colab Secrets named HF_TOKEN
import os
from google.colab import userdata

HF_TOKEN = os.environ.get("HF_TOKEN") or ""
try:
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

assert HF_TOKEN, "Set HF_TOKEN (HF write token) before running"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

HF_USER = "rssebambulidde"
HF_REPO = f"{HF_USER}/adtc-kilimo-0.5b-gguf"
GGUF_NAME = "adtc-kilimo-0.5b-q4_k_m.gguf"
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
print("HF repo target:", HF_REPO)

In [ ]:
!nvidia-smi
%pip install -q "transformers>=4.44" "peft>=0.12" "trl>=0.9" "datasets>=2.19" accelerate safetensors pyyaml huggingface_hub sentencepiece protobuf
%pip install -q "llama-cpp-python==0.3.4" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 || true

In [ ]:
# Clone submission repo (public or your fork). Override if private.
REPO_URL = "https://github.com/rssebambulidde/adtc-2026.git"
import os, pathlib
if not pathlib.Path("adtc-2026").exists():
    !git clone --depth 1 {REPO_URL} adtc-2026
%cd adtc-2026
!python scripts/build_dataset.py --out data/build/train.jsonl --augment
!wc -l data/build/train.jsonl

In [ ]:
!python scripts/train_lora.py --config configs/kilimo-0.5b.yaml

In [ ]:
# Merge LoRA → full HF weights
import torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

adapter = "checkpoints/kilimo-0.5b-lora"
merged = Path("merged/kilimo-0.5b")
merged.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(adapter, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True
)
model = PeftModel.from_pretrained(model, adapter)
model = model.merge_and_unload()
model.save_pretrained(merged, safe_serialization=True)
tokenizer.save_pretrained(merged)
print("merged ->", merged)

In [ ]:
# Convert + quantize with llama.cpp
%cd /content
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!cmake -B build -DGGML_CUDA=OFF
!cmake --build build --config Release -j --target llama-quantize
%pip install -q -r requirements.txt

!python convert_hf_to_gguf.py /content/adtc-2026/merged/kilimo-0.5b \
  --outfile /content/adtc-2026/model/adtc-kilimo-0.5b-f16.gguf --outtype f16

!./build/bin/llama-quantize \
  /content/adtc-2026/model/adtc-kilimo-0.5b-f16.gguf \
  /content/adtc-2026/model/adtc-kilimo-0.5b-q4_k_m.gguf Q4_K_M

!ls -lh /content/adtc-2026/model/*.gguf
!sha256sum /content/adtc-2026/model/adtc-kilimo-0.5b-q4_k_m.gguf

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
path = f"/content/adtc-2026/model/{GGUF_NAME}"
api.upload_file(
    path_or_fileobj=path,
    path_in_repo=GGUF_NAME,
    repo_id=HF_REPO,
    token=HF_TOKEN,
)
readme = f"""---
license: apache-2.0
base_model: {BASE}
tags:
- gguf
- agriculture
- adtc-2026
---
# ADTC Kilimo 0.5B (Q4_K_M)

LoRA-tuned Qwen2.5-0.5B-Instruct for East African smallholder agriculture advisory.\n
File: `{GGUF_NAME}`\n
"""
api.upload_file(
    path_or_fileobj=readme.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO,
    token=HF_TOKEN,
)
import hashlib, pathlib
h = hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()
print("Uploaded:", HF_REPO)
print("EXPECTED_SHA256=", h)
print("Set in download_model.sh: MODEL_REPO=", HF_REPO)